In [1]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Project config
from src.params import *

# Plot configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# General outline for EDA 

### Phase I: Data Quality
**To look at:**
- Overall aspect of sample PM2.5 curves
- % day availability per city/sensor
- Number, max length and distribution across the train set of missing days
- time distribution of sensors
- Distribution of intra-day coverage per sensor
- Aberrant values (high or low), frequency and distribution across train set

**Output:**
- Threshold for filtering out sensors with day availability below x% of total days
- Decision on max gap length acceptable (interpolation vs suppression)
- List of cities to remove because insufficient data quality
- Decision on handling aberrant data
- First city reduction based on data quality (target: 5-7 cities from 12)


### Phase II: Sensor Metrics and Aggregation
**To look at:**
- Correlation between same-city sensors
- Inter-sensor variance
- Temporal stability of sensors

**Output:**
- Coverage threshold per day for filtering out unreliable sensors (if needed)
- min sensors to keep a city
- Rule of aggregation: simple average? weighted average by coverage?
- further reduction of city based on sensors availability

### Phase III: Target Distribution
**To look at:**
- Shape of PM2.5 distributions (after sensor aggregation)
- Distribution of values 
- Outliers detection and frequency

**Output:**
- Decision on log transformation
- Choice of metric: RMSE vs RMSLE vs MAE 


### Phase IV: Temporal Patterns
**To look at:**
- Trend over time
- Seasonality (monthly patterns)
- Distribution of day-to-day changes
- Volatility: rolling std sur 7d/30d 

**Output:**
- Knowledge about general behavior, extreme periods
- Decision on retraining frequency 


### Phase V: Autocorrelation
**To look at:**
- ACF/PACF plots per city
- Identification of significant lags

**Output:**
- Relevant lookback window (e.g., 7 days, 14 days, 30 days)
- List of lags to include as features (e.g., all individual lags from 1 to 7, or specific lags: 1, 7, 14, 30)
- Alternative: decision on using rolling features (mean/std) vs individual lags


### Phase VI: Inter-City Comparison
**To look at:**
- Value distributions between cities
- Temporal patterns comparison across cities
- Cross-correlation between cities (pattern similarity)

**Output:**
- Decision on model architecture: 1 global model + city feature OR separate models per city
- Final city selection: keep 5-7 cities with best data quality 


# merging weather and air qual 

In [2]:
df_weather = pd.read_csv("../data/raw/weather.csv")
df_airqual = pd.read_csv("../data/raw/aq_data.csv")

In [3]:
df_weather.head(3)

,date,temp_min,temp_max,temp_avg,cloud_cover,humidity,precipitation,pressure,wind_speed,wind_direction,city
0,2023-05-01,9.72,16.09,12.905,100.0,79.0,1.89,1018.0,6.69,330.0,Paris
1,2023-05-02,7.44,18.19,12.815,20.0,74.0,0.00,1025.0,5.14,60.0,Paris
2,2023-05-03,7.71,21.17,14.440,0.0,58.0,0.00,1023.0,6.69,90.0,Paris


In [4]:
#check that no missing days in weather df
df_weather["date"].nunique()/(df_weather.shape[0] / df_weather["city"].nunique())

1.0

In [5]:
#convert date in pd datetime format
df_weather["date"] = pd.to_datetime(df_weather["date"])

In [6]:
df_airqual.head(2)

,sensor_id,date_from_utc,date_from_local,date_to_utc,date_to_local,pm25_avg,pm25_min,pm25_q25,pm25_median,pm25_q75,pm25_max,coverage,city
0,1582598,2023-04-30T22:00:00Z,2023-05-01T00:00:00+02:00,2023-05-01T22:00:00Z,2023-05-02T00:00:00+02:00,16.0,4.9,10.675,15.05,19.65,32.0,100.0,Paris
1,1582598,2023-05-01T22:00:00Z,2023-05-02T00:00:00+02:00,2023-05-02T22:00:00Z,2023-05-03T00:00:00+02:00,12.6,6.6,9.650,10.90,15.60,21.9,100.0,Paris


In [7]:
df_airqual["date"] = pd.to_datetime(df_airqual["date_from_local"].str[:10])

In [10]:
#merge df matching cities and date, leaving any missing day as NaN for further investigation
df_merged = pd.merge(df_airqual, df_weather, on=["date", "city"], how= "outer")

In [14]:
df_merged.columns
new_order = ["city", "sensor_id","date", 'pm25_avg', 'pm25_min', 'pm25_q25', 'pm25_median',
       'pm25_q75', 'pm25_max', 'coverage','temp_min',
       'temp_max', 'temp_avg', 'cloud_cover', 'humidity', 'precipitation',
       'pressure', 'wind_speed', 'wind_direction' ]
df_merged = df_merged[new_order]

,city,sensor_id,date,pm25_avg,pm25_min,pm25_q25,pm25_median,pm25_q75,pm25_max,coverage,temp_min,temp_max,temp_avg,cloud_cover,humidity,precipitation,pressure,wind_speed,wind_direction
0,Barcelona,4273113.0,2023-05-01,7.0,7.00,7.00,7.000,7.0000,7.000000,4.0,15.96,19.51,17.735,40.0,76.0,0.00,1017.0,6.69,110.0
1,Berlin,1300115.0,2023-05-01,12.9,8.09,9.74,10.950,14.0300,27.889999,100.0,4.70,19.37,12.035,0.0,38.0,0.00,1015.0,4.92,104.0
2,Berlin,1300119.0,2023-05-01,13.0,8.21,10.30,12.055,13.9250,24.610001,100.0,4.70,19.37,12.035,0.0,38.0,0.00,1015.0,4.92,104.0
3,Berlin,1300111.0,2023-05-01,14.0,9.25,11.09,12.120,13.7525,29.379999,100.0,4.70,19.37,12.035,0.0,38.0,0.00,1015.0,4.92,104.0
4,Delhi,23534.0,2023-05-01,44.2,21.00,36.00,44.500,52.0000,67.000000,200.0,20.09,24.09,22.090,75.0,78.0,2.65,1009.0,3.73,117.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23066,Rome,21846.0,2025-04-30,12.0,12.00,12.00,12.000,12.0000,12.000000,4.0,14.31,23.85,19.080,0.0,53.0,0.00,1019.0,4.12,250.0
23067,Rome,21841.0,2025-04-30,10.0,10.00,10.00,10.000,10.0000,10.000000,4.0,14.31,23.85,19.080,0.0,53.0,0.00,1019.0,4.12,250.0
23068,Rome,21857.0,2025-04-30,10.0,10.00,10.00,10.000,10.0000,10.000000,4.0,14.31,23.85,19.080,0.0,53.0,0.00,1019.0,4.12,250.0
23069,Santiago,3190.0,2025-04-30,34.7,31.00,33.00,34.000,35.2500,39.000000,100.0,10.21,16.03,13.120,92.0,68.0,0.00,1015.0,2.57,150.0


##